In [ ]:
import pandas as pd
import numpy as np

# === Load ===
apc           = pd.read_csv('../data/raw/apc.csv', sep=';', encoding='latin-1')
matched_stops = pd.read_csv('../data/raw/matched_stops.csv', sep=';', decimal=',', encoding='latin-1')

apc['arrival_dt']           = pd.to_datetime(apc['arrival'], format='%d/%m/%Y %H:%M:%S')
matched_stops['arrival_dt'] = pd.to_datetime(matched_stops['arrivalTime'], format='%m-%d-%Y %I:%M:%S %p')

print(f"apc rows:           {len(apc):,}")
print(f"matched_stops rows: {len(matched_stops):,}")

# === Match ===
time_tol_sec = 60
gps_tol      = 0.0020

apc_sorted = apc.sort_values('arrival_dt').reset_index(drop=True)
apc_times    = apc_sorted['arrival_dt'].values
apc_lats     = apc_sorted['latitude'].values
apc_lons     = apc_sorted['longitude'].values
apc_vehicles = apc_sorted['vehicleCode'].values

n_matches      = np.zeros(len(matched_stops), dtype=int)
matched_apc_v  = np.full(len(matched_stops), -1, dtype=int)

for i in range(len(matched_stops)):
    ms_row = matched_stops.iloc[i]
    t = ms_row['arrival_dt'].to_datetime64()
    
    left  = np.searchsorted(apc_times, t - np.timedelta64(time_tol_sec, 's'))
    right = np.searchsorted(apc_times, t + np.timedelta64(time_tol_sec, 's'))
    
    if left == right:
        continue
    
    lat_ok = np.abs(apc_lats[left:right] - ms_row['latitude'])  <= gps_tol
    lon_ok = np.abs(apc_lons[left:right] - ms_row['longitude']) <= gps_tol
    in_window = lat_ok & lon_ok
    n = int(in_window.sum())
    n_matches[i] = n
    
    if n == 1:
        matched_apc_v[i] = apc_vehicles[left:right][in_window][0]

matched_stops['n_matches']   = n_matches
matched_stops['apc_vehicle'] = matched_apc_v

print()
print(f"0 matches:        {(n_matches == 0).sum():,}")
print(f"1 match (unique): {(n_matches == 1).sum():,}")
print(f"2+ matches:       {(n_matches >= 2).sum():,}")

# === Keep only unique matches ===
unique = matched_stops[matched_stops['n_matches'] == 1].copy()
print(f"\nKept {len(unique):,} unique matches")

# === Vehicle mapping ===
mapping_rows = []
for dilax_v, grp in unique.groupby('vehicleNumber'):
    counts = grp['apc_vehicle'].value_counts()
    mapping_rows.append({
        'dilax_vehicle': dilax_v,
        'apc_vehicle':   counts.index[0],
        'n_links':       int(counts.sum()),
        'confidence':    counts.iloc[0] / counts.sum(),
        'distinct_apc':  len(counts),
    })

mapping = pd.DataFrame(mapping_rows).sort_values('dilax_vehicle').reset_index(drop=True)

print(f"\nUnique Dilax vehicles in linked data: {unique['vehicleNumber'].nunique()}")
print(f"\n=== Vehicle mapping ===")
print(mapping.to_string(index=False))

# === Save ===
unique[['arrival_dt', 'vehicleNumber', 'apc_vehicle', 'pointShortLabel', 'stopLabel', 'tripNumber']].to_csv(
    '../data/processed/matched_stops_linked.csv', sep=';', index=False, encoding='latin-1')
mapping.to_csv('../data/processed/vehicle_mapping.csv', sep=';', decimal=',', index=False, encoding='latin-1')

print("\nSaved:")
print("  ../data/processed/matched_stops_linked.csv")
print("  ../data/processed/vehicle_mapping.csv")

apc rows:           1,598,279
matched_stops rows: 56,165

0 matches:        4,028
1 match (unique): 46,985
2+ matches:       5,152

Kept 46,985 unique matches

Unique Dilax vehicles in linked data: 23

=== Vehicle mapping ===
 dilax_vehicle  apc_vehicle  n_links  confidence  distinct_apc
            90         4288       28    0.964286             2
           106         4259      251    0.980080             4
           117         4270        4    0.500000             2
           133         4205     3649    0.976980            15
           138         4197    17028    0.982265            41
           204         4213      700    0.964286             9
           216         4222     1981    0.987885             7
           219         4226     3003    0.973360            13
           222         4212      321    1.000000             1
           224         4240     5334    0.985189            17
           240         4227     5951    0.975970            22
           244    